In [ ]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import plotly.express as px
sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import apply_conservative_classification
from itertools import product


In [ ]:
ss = [
    {'solution_folder': f"RTS-GMLC_v32.3s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v32.1s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v25.1s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v11.1.0s", 'model_type' : 'envelope'},
    ]

In [ ]:

s_ = "s_uc"
days = range(1,365)
scalar = []
scalar_ =pd.DataFrame()
for sol in ss:
    s = sol['solution_folder']
    for day in days:
        try:
            scalar_ = pd.read_parquet(os.path.join("..", "output", s, f'n_{day}', f'{s_}_scalar.parquet'))
            print(f'(ss, days):{s}, n_{day}')
            scalar_['day'] = day
            scalar_['solution_id'] = sol['solution_folder']
            scalar_['model_type'] = sol['model_type']
            scalar.append(scalar_)
        except Exception:
            pass
    
    days_str = "-".join(str(d) for d in days)
    try:
        scalar_ = pd.read_parquet(os.path.join("..", "output", s, f'n_{days_str}', f'{s_}_scalar.parquet'))
        print(f'(ss, days):{s}, n_{days_str}')
        # scalar_['day'] = 0
        scalar_['solution_id'] = sol['solution_folder']
        scalar_['model_type'] = sol['model_type']
        scalar.append(scalar_)
    except Exception:
        pass

scalar = apply_conservative_classification(pd.concat(scalar))        

In [ ]:
filter_ = scalar.pivot(
    index='day',
    columns = 'model_type',
).dropna()

missing_days = [d for d in range(1, 365) if d not in filter_.index.unique()]
print("Missing days:", missing_days)
print("Number of missing days:", len(missing_days))

In [ ]:
px.bar(scalar.groupby(['solution_id','termination_status_discrete_model']).count().reset_index(), color = 'termination_status_discrete_model', y='day', x ='solution_id', barmode = 'group')

In [ ]:
fig = px.scatter(scalar, y='relative_gap_discrete_model', x='day', color='termination_status_discrete_model', facet_col='solution_id')

fig.add_hline(y=0.0001, line_dash="dash", line_color="green")
fig.show()

In [ ]:
scalar

In [ ]:
fig = px.scatter(scalar, y='relative_gap_discrete_model', x='solve_time', color='termination_status_discrete_model', title=f'Solve time vs days ({s_})', facet_col='solution_id', hover_data=['day'])
fig.show()

In [ ]:
scalar[scalar.termination_status_discrete_model == 'OPTIMAL'].groupby('model_type')['solve_time'].describe()
scalar.groupby('model_type')['solve_time'].describe().round(1)